# Load data

In [1]:
import pandas as pd

In [2]:
selected_data = pd.read_pickle(r"..\..\..\data\preprocessed\full_data_organizations.pkl")

# Hugging Face (*Transformers*) 

In [3]:
from transformers import pipeline, BertTokenizer

Import model

In [4]:
distilled_student_sentiment_classifier = pipeline(
    model="lxyuan/distilbert-base-multilingual-cased-sentiments-student", 
    return_all_scores=True
)

c:\Users\eduardo.moreno\AppData\Local\anaconda3\envs\env_nlp\Lib\site-packages\transformers\pipelines\text_classification.py:104: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


Import tokenizer

In [5]:
tokenizer = BertTokenizer.from_pretrained('lxyuan/distilbert-base-multilingual-cased-sentiments-student')

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'DistilBertTokenizer'. 
The class this function is called from is 'BertTokenizer'.


Example of use and response of the model

In [6]:
distilled_student_sentiment_classifier("I love this movie and i would watch it again and again!")

[[{'label': 'positive', 'score': 0.9731044769287109},
  {'label': 'neutral', 'score': 0.016910076141357422},
  {'label': 'negative', 'score': 0.009985473938286304}]]

Let's use it with our data

In [7]:
print(selected_data.iloc[418233]['Message'])

Al celebrarse hoy, 5 de junio, el Día Mundial del Medio Ambiente desde el 1954 el Ministerio de Defensa, sus instituciones y dependencias, reafirma su apoyo incondicional a la conservación del medio ambiente en toda su área de responsabilidad en el territorio de la República Dominicana, sus espacios terrestres, marítimos y aéreos. El MIDE se une hoy al Ministerio de Medio Ambiente y Recursos Naturales, razón por la cual seguimos realizando operaciones en todo el territorio nacional para proteger todos los ecosistemas dominicanos. Luchamos contra la tala de árboles para conservar las cuencas acuíferas e hidrográficas que dan origen a los ríos y arroyos que proveen al pueblo dominicano del preciado liquido, como es agua potable. De igualmente manera, ante los retos que nos impone el cambio climático, dentro de nuestra planificación estratégica está concebida la conservación de energía así como el fomento para el uso de fuentes alternativas como la energía solar. El Día Mundial del Medio 

In [8]:
distilled_student_sentiment_classifier(selected_data.iloc[418233]['Message'])

[[{'label': 'positive', 'score': 0.6169759035110474},
  {'label': 'neutral', 'score': 0.07673612982034683},
  {'label': 'negative', 'score': 0.3062879145145416}]]

`distilbert-base-multilingual-cased-sentiments-student` as it base model `bert` accepts only $512$ tokens, so what do you think will happen if we try to analyze an input **greater than 512 tokens**?


In [9]:
txt = selected_data['Message'][0]
print(txt)

A LOS NACIDOS ENTRE 1970 Y 1987 Somos una generación especial y nos denominaron la generación X y como no, si nuestra infancia estuvo llena de cambios Somos la última generación que jugaba en la calle y en los recreos del colegio a las bolitas, a el escondite, somos la primera generación que jugó con videojuegos, fuimos a parques de atracciones y vimos caricaturas a color. Fuimos los últimos en grabar canciones de la radio en casettes (como olvidarlo si mientras se grababa en algunos casos no podíamos ni hablar) y vimos películas en versión Beta y VHS PERO orgullosos pioneros del personal stereo y los CD's. Cuantos no tuvimos que tragarnos, Salvado por la Campana (con todo y Screech), Beverly Hills 90210. Nosotros vimos la caída de torres gemelas y también vimos caer el muro de Berlín. Aprendimos a utilizar los ordenadores antes que nuestros padres y abuelos, y sobre todo antes de todos esos niños cerebritos de hoy en día y nunca vimos a los que no sabían usar los ordenadores como una 

In [10]:
tokens = tokenizer.encode_plus(txt, add_special_tokens=False)

len(tokens['input_ids'])

Token indices sequence length is longer than the specified maximum sequence length for this model (1201 > 512). Running this sequence through the model will result in indexing errors


1201

In [11]:
tokens = tokenizer.encode_plus(txt, add_special_tokens=False,
                               return_tensors='pt')

print(len(tokens['input_ids'][0]))
tokens

1201


{'input_ids': tensor([[  138,   149, 21793,  ..., 12816, 12882, 10466]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1]])}

Here we have specified a few arguments that require some explanation.

* `max_length` - this tell the tokenizer the maximum number of tokens we want to see in each sample, for BERT we almost always use `512` as that is the length of sequences that BERT consumes.

* `truncation` - if our input string `txt` contains more tokens than allowed (specified in `max_length` parameter) then we cut all tokens past the `max_length` limit.

* `padding` - if our input string `txt` contains less tokens than specified by `max_length` then we pad the sequence with zeros (`0` is the token ID for *'[PAD]'* - BERTs padding token).

* `add_special_tokens` - whether or not to add special tokens, when using BERT we always want this to be `True` unless we are adding them ourselves.

| Token | ID | Description |
| --- | --- | --- |
| [PAD] | 0 | Used to fill empty space when input sequence is shorter than required sequence size for model |
| [UNK] | 100 | If a word/character is not found in BERTs vocabulary it will be represented by this *unknown* token |
| [CLS] | 101 | Represents the start of a sequence |
| [SEP] | 102 | Seperator token to denote the end of a sequence and as a seperator where there are multiple sequences |
| [MASK] | 103 | Token used for masking other tokens, used for masked language modeling |

*Note that our tokenized sequence begins with `101`, the seperator token `102` can be found seperating the input sequence and padding tokens `0`.*

* `return_tensors` - here we specify either `'pt'` to return PyTorch tensors, or `'tf'` to return TensorFlow tensors.

In [12]:
try:
    response = distilled_student_sentiment_classifier(txt)
    print(response)
except Exception as e:
    print(f"Something went WRONG: {e}")

Token indices sequence length is longer than the specified maximum sequence length for this model (1203 > 512). Running this sequence through the model will result in indexing errors


Something went WRONG: The size of tensor a (1203) must match the size of tensor b (512) at non-singleton dimension 1


What happen if we only send a chunk of the post?

In [13]:
try:
    response = distilled_student_sentiment_classifier(txt[:1500])
    print(response)
except Exception as e:
    print(f"Something went WRONG: {e}")

[[{'label': 'positive', 'score': 0.28041714429855347}, {'label': 'neutral', 'score': 0.1233644187450409}, {'label': 'negative', 'score': 0.5962184071540833}]]


We will use [LangChain](https://python.langchain.com/v0.2/docs/introduction/) to split posts so we can compute the sentiment of each split.

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [15]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,  # Ajusta según sea necesario
    chunk_overlap=50,
    separators=[" "]
)

In [16]:
texts = text_splitter.create_documents([txt])
print(len(texts))

7


In [17]:
len(texts[0].page_content)

693

In [18]:
texts[0].page_content[:100]

'A LOS NACIDOS ENTRE 1970 Y 1987 Somos una generación especial y nos denominaron la generación X y co'

In [19]:
texts[0].page_content[-115:]

"eros del personal stereo y los CD's. Cuantos no tuvimos que tragarnos, Salvado por la Campana (con todo y Screech),"

In [20]:
texts[1].page_content[100:]

'gemelas y también vimos caer el muro de Berlín. Aprendimos a utilizar los ordenadores antes que nuestros padres y abuelos, y sobre todo antes de todos esos niños cerebritos de hoy en día y nunca vimos a los que no sabían usar los ordenadores como una especie de "retardados" como sucede hoy. Jugamos con el Double Dragon, Street Fighter, empezamos el Mortal Kombat, el tetris, el Mario Bross, nos pegábamos a la tele a mirar Hugo y como otros jugaban desde sus casas usando el teléfono de red fija!! vimos los anuncios de los primeros celulares (que parecían ladrillos) y creímos que Internet sería'

In [21]:
pos_sent = []
neu_sent = []
neg_sent = []

for text_i in texts:
    response = distilled_student_sentiment_classifier(text_i.page_content)

    print(text_i.page_content)
    print(response, '\n')

    for sent in response[0]:
        if sent['label'] == 'positive':
            pos_sent.append(sent['score'])
        elif sent['label'] == 'neutral':
            neu_sent.append(sent['score'])
        elif sent['label'] == 'negative':
            neg_sent.append(sent['score'])

A LOS NACIDOS ENTRE 1970 Y 1987 Somos una generación especial y nos denominaron la generación X y como no, si nuestra infancia estuvo llena de cambios Somos la última generación que jugaba en la calle y en los recreos del colegio a las bolitas, a el escondite, somos la primera generación que jugó con videojuegos, fuimos a parques de atracciones y vimos caricaturas a color. Fuimos los últimos en grabar canciones de la radio en casettes (como olvidarlo si mientras se grababa en algunos casos no podíamos ni hablar) y vimos películas en versión Beta y VHS PERO orgullosos pioneros del personal stereo y los CD's. Cuantos no tuvimos que tragarnos, Salvado por la Campana (con todo y Screech),
[[{'label': 'positive', 'score': 0.45823341608047485}, {'label': 'neutral', 'score': 0.15719184279441833}, {'label': 'negative', 'score': 0.3845748007297516}]] 

Salvado por la Campana (con todo y Screech), Beverly Hills 90210. Nosotros vimos la caída de torres gemelas y también vimos caer el muro de Berl

For the final output we will compute the average of sentiment on each partition and will return the highest as the final sentiment of the initial post.

In [22]:
import numpy  as np

In [23]:
mean_scores = [np.mean(pos_sent), np.mean(neu_sent), np.mean(neg_sent)]

In [24]:
mean_scores

[0.4076317846775055, 0.14786760455795697, 0.44450063684156965]

In [25]:
sentiment = np.argmax(mean_scores)
score = np.max(mean_scores)

if sentiment == 0:
    print(f"The sentiment of the post is POSITIVE: {score}")
if sentiment == 1:
    print(f"The sentiment of the post is NEUTRAL: {score}")
if sentiment == 2:
    print(f"The sentiment of the post is NEGATIVE: {score}")

The sentiment of the post is NEGATIVE: 0.44450063684156965


Now let's compute the sentiment for each post!

In [26]:
from tqdm import tqdm

In [27]:
def get_sentiment(post):
    try:
        response = distilled_student_sentiment_classifier(post)
        pos_sent = []
        neu_sent = []
        neg_sent = []
        for sent in response[0]:
            if sent['label'] == 'positive':
                pos_sent.append(sent['score'])
            elif sent['label'] == 'neutral':
                neu_sent.append(sent['score'])
            elif sent['label'] == 'negative':
                neg_sent.append(sent['score'])
        mean_scores = [np.mean(pos_sent), np.mean(neu_sent), np.mean(neg_sent)]
        sentiment = np.argmax(mean_scores)
        score = np.max(mean_scores)
        if sentiment == 0:
            sentiment = 'Positive'
        elif sentiment == 1:
            sentiment = 'Neutral'
        else:
            sentiment = 'Negative'

    #except Exception as e:
    except BaseException as e:
        # Split the post
        texts = text_splitter.create_documents([post])
        # Compute the sentiment for each split
        pos_sent = []
        neu_sent = []
        neg_sent = []
        for text_i in texts:
            response = distilled_student_sentiment_classifier(text_i.page_content)
            # Compute avg sentiment
            for sent in response[0]:
                if sent['label'] == 'positive':
                    pos_sent.append(sent['score'])
                elif sent['label'] == 'neutral':
                    neu_sent.append(sent['score'])
                elif sent['label'] == 'negative':
                    neg_sent.append(sent['score'])
        # Compute final sentiment and score
        mean_scores = [np.mean(pos_sent), np.mean(neu_sent), np.mean(neg_sent)]
        sentiment = np.argmax(mean_scores)
        score = np.max(mean_scores)
        if sentiment == 0:
            sentiment = 'Positive'
        elif sentiment == 1:
            sentiment = 'Neutral'
        else:
            sentiment = 'Negative'
    
    return {'sentiment': sentiment, 'score': score}


In [28]:
def get_sentiment(post):

    tokens = distilled_student_sentiment_classifier.tokenizer(post, return_tensors='pt', truncation=False)['input_ids']

    if tokens.size(1) < 512:
        response = distilled_student_sentiment_classifier(post)
        pos_sent = []
        neu_sent = []
        neg_sent = []
        for sent in response[0]:
            if sent['label'] == 'positive':
                pos_sent.append(sent['score'])
            elif sent['label'] == 'neutral':
                neu_sent.append(sent['score'])
            elif sent['label'] == 'negative':
                neg_sent.append(sent['score'])
        mean_scores = [np.mean(pos_sent), np.mean(neu_sent), np.mean(neg_sent)]
        sentiment = np.argmax(mean_scores)
        score = np.max(mean_scores)
        if sentiment == 0:
            sentiment = 'Positive'
        elif sentiment == 1:
            sentiment = 'Neutral'
        else:
            sentiment = 'Negative'

    else:
        # Split the post
        texts = text_splitter.create_documents([post])
        # Compute the sentiment for each split
        pos_sent = []
        neu_sent = []
        neg_sent = []
        for text_i in texts:
            response = distilled_student_sentiment_classifier(text_i.page_content)
            # Compute avg sentiment
            for sent in response[0]:
                if sent['label'] == 'positive':
                    pos_sent.append(sent['score'])
                elif sent['label'] == 'neutral':
                    neu_sent.append(sent['score'])
                elif sent['label'] == 'negative':
                    neg_sent.append(sent['score'])
        # Compute final sentiment and score
        mean_scores = [np.mean(pos_sent), np.mean(neu_sent), np.mean(neg_sent)]
        sentiment = np.argmax(mean_scores)
        score = np.max(mean_scores)
        if sentiment == 0:
            sentiment = 'Positive'
        elif sentiment == 1:
            sentiment = 'Neutral'
        else:
            sentiment = 'Negative'
    
    return {'sentiment': sentiment, 'score': score}


We will select a sample from all data and compute the sentiment for those posts.

In [29]:
import random
from random import sample, seed

In [30]:
seed(42)
sub_sample = selected_data.loc[sample(selected_data.index.to_list(), int(len(selected_data.index.to_list())*0.1))]

Aditionally we will remove URLs from all posts

In [31]:
import re

In [32]:
def remove_urls(text):
    # Expresión regular para encontrar URLs
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    # Reemplaza todas las URLs con una cadena vacía
    return url_pattern.sub('', text)

In [33]:
sub_sample['Message_clean'] = sub_sample['Message'].apply(lambda x: remove_urls(x))

In [34]:
sub_sample

,Message,Message_stpWrd,Message_clean_lemm_stpWrd,organizations,Message_clean
375233,PLAN DE EMERGENCIA Y CONTINGENCIA DISEÑADO POR...,PLAN EMERGENCIA CONTINGENCIA DISEÑADO ADMINIST...,plan emergencia contingencia disenado administ...,"[BOGOTÁ, Cambio Climático]",PLAN DE EMERGENCIA Y CONTINGENCIA DISEÑADO POR...
83975,un análisis breve sobre la #COP21,análisis breve #COP21,analisis breve cop21,[],un análisis breve sobre la #COP21
774421,RT. Un docena de manifestantes semidesnudos de...,RT. docena manifestantes semidesnudos movimien...,rt docena manifestante semidesnudo movimiento ...,"[RT, Parlamento británico]",RT. Un docena de manifestantes semidesnudos de...
699239,Agregó que también se darán a conocer las sanc...,darán sanciones cumplan reglamento.,agregar tambien sancion cumplir reglamento,[],Agregó que también se darán a conocer las sanc...
646095,⚠️ATENTO AVISO⚠️ Debido a varias afectaciones ...,⚠️ATENTO AVISO⚠️ afectaciones tormenta inverna...,atento aviso afectacion tormenta invernal ciud...,[COP 25 sobre Cambio Climático],⚠️ATENTO AVISO⚠️ Debido a varias afectaciones ...
...,...,...,...,...,...
299552,En Colombia se previenen todos los riesgos a l...,Colombia previenen riesgos salud cambio climát...,colombia prevenir riesgo salud cambio climatic...,[],En Colombia se previenen todos los riesgos a l...
1228502,🔵 Medidas del PPCV en el Debate de Política Ge...,🔵 Medidas PPCV Debate Política General. ✅ Plan...,medida ppcv debate politica general plan ppcv ...,"[PPCV, Cambio Climático #DebatCV]",🔵 Medidas del PPCV en el Debate de Política Ge...
623413,#GrandesLogros2019 | El estado de deterioro de...,#GrandesLogros2019 | deterioro planeta 🌎 movil...,grandeslogros2019 deterioro planeta movilizar ...,[],#GrandesLogros2019 | El estado de deterioro de...
405585,"#JPCCinforma Los movimientos NuestrasVoces, Gr...","#JPCCinforma movimientos NuestrasVoces, GreenF...",jpccinforma movimiento nuestrasvoz greenfaith ...,"[Consejo Episcopal Latinoamericano, Movimiento...","#JPCCinforma Los movimientos NuestrasVoces, Gr..."


In [35]:
list_msgs = sub_sample['Message_clean'].to_list()

Next cell took around **6 hours** to complete (please be patient)...

In [36]:
if 1 == 0:
    list_sentiment = []
    list_scores = []

    for post_i in tqdm(list_msgs):
        response = get_sentiment(post_i)
        list_sentiment.append(response['sentiment'])
        list_scores.append(response['score'])
    
    sub_sample['sentiment'] = list_sentiment
    sub_sample['score'] = list_scores

    sub_sample.to_pickle(r"..\..\..\data\preprocessed\sentiment_analysis_subsample.pkl")
else:
    sub_sample = pd.read_pickle(r"..\..\..\data\preprocessed\sentiment_analysis_subsample.pkl")

In [37]:
sub_sample.head()

,Message,Message_stpWrd,Message_clean_lemm_stpWrd,organizations,Message_clean,sentiment,score
375233,PLAN DE EMERGENCIA Y CONTINGENCIA DISEÑADO POR...,PLAN EMERGENCIA CONTINGENCIA DISEÑADO ADMINIST...,plan emergencia contingencia disenado administ...,"[BOGOTÁ, Cambio Climático]",PLAN DE EMERGENCIA Y CONTINGENCIA DISEÑADO POR...,Positive,0.681693
83975,un análisis breve sobre la #COP21,análisis breve #COP21,analisis breve cop21,[],un análisis breve sobre la #COP21,Positive,0.482537
774421,RT. Un docena de manifestantes semidesnudos de...,RT. docena manifestantes semidesnudos movimien...,rt docena manifestante semidesnudo movimiento ...,"[RT, Parlamento británico]",RT. Un docena de manifestantes semidesnudos de...,Negative,0.819510
699239,Agregó que también se darán a conocer las sanc...,darán sanciones cumplan reglamento.,agregar tambien sancion cumplir reglamento,[],Agregó que también se darán a conocer las sanc...,Negative,0.530870
646095,⚠️ATENTO AVISO⚠️ Debido a varias afectaciones ...,⚠️ATENTO AVISO⚠️ afectaciones tormenta inverna...,atento aviso afectacion tormenta invernal ciud...,[COP 25 sobre Cambio Climático],⚠️ATENTO AVISO⚠️ Debido a varias afectaciones ...,Negative,0.627384


In [38]:
print(sub_sample['Message'][646095])

⚠️ATENTO AVISO⚠️ Debido a varias afectaciones por la tormenta invernal en la ciudad de Tijuana, oficialmente se pospone la 4ta Manifestación Mundial por la Crisis Climática para el sábado 7 de Diciembre en el marco de la COP 25 sobre Cambio Climático. Ayudanos a compartir para que más personas se enteren. 🙏🌎


In [39]:
sub_sample[sub_sample['sentiment'] == 'Positive'].sort_values(by='score', ascending=False)

,Message,Message_stpWrd,Message_clean_lemm_stpWrd,organizations,Message_clean,sentiment,score
297546,Excelente iniciativa! Sumémosnos!,Excelente iniciativa! Sumémosnos!,excelente iniciativa sumemosno,[],Excelente iniciativa! Sumémosnos!,Positive,0.994401
256522,Una buena iniciativa. Felicitaciones a la @CAM...,iniciativa. Felicitaciones @CAMHUILA propuesta...,iniciativa felicitacion camhuila propuesta edu...,[@CAMHUILA],Una buena iniciativa. Felicitaciones a la @CAM...,Positive,0.994268
787423,"Que gran noticia, una cosa buenísima para el p...","noticia, cosa buenísima planeta",noticia cosa buenisimo planeta,[],"Que gran noticia, una cosa buenísima para el p...",Positive,0.992874
234216,¡Qué buena campaña! #PremiosEscobillaDeOro 10 ...,¡Qué campaña! #PremiosEscobillaDeOro 10 multin...,campana premiosescobilladeoro 10 multinacional...,[],¡Qué buena campaña! #PremiosEscobillaDeOro 10 ...,Positive,0.992004
1101783,"Hermosa iniciativa, gracias por educar a nuest...","Hermosa iniciativa, gracias educar niños.",hermoso iniciativa gracia educar nino,[],"Hermosa iniciativa, gracias por educar a nuest...",Positive,0.991898
...,...,...,...,...,...,...,...
329308,"Aumento de temperatura, incremento de niveles ...","Aumento temperatura, incremento niveles CO2 su...",aumento temperatura incremento nivel co2 subid...,[NASA],"Aumento de temperatura, incremento de niveles ...",Positive,0.345049
606307,Mañana a las 11:30 am no te puedes perder la e...,Mañana 11:30 am puedes perder entrevista harem...,manana 1130 am perder entrevista ivan lanegra ...,[],Mañana a las 11:30 am no te puedes perder la e...,Positive,0.342982
67766,Hay 3 cursos ebook de CEATECI para hacer negoc...,3 cursos ebook CEATECI negocios DAÑAR medio am...,3 curso ebook ceateci negocio danar medio ambi...,[TECNICOS],Hay 3 cursos ebook de CEATECI para hacer negoc...,Positive,0.342390
1267959,La alteración de temperaturas que se presentan...,alteración temperaturas presentan Sonora compo...,alteracion temperatura presentar sonoro compor...,[],La alteración de temperaturas que se presentan...,Positive,0.342380


In [40]:
sub_sample[sub_sample['sentiment'] == 'Negative'].sort_values(by='score', ascending=False)

,Message,Message_stpWrd,Message_clean_lemm_stpWrd,organizations,Message_clean,sentiment,score
666466,"“…Es peor de lo que uno imaginaba, es mucho pe...","“…Es peor imaginaba, peor”: Análisis efecto ca...",peor imaginar peor analisis efecto calentamien...,[],"“…Es peor de lo que uno imaginaba, es mucho pe...",Negative,0.983568
498761,"""Es un día triste para la comunidad global"": l...","""Es día triste comunidad global"": líderes inte...",dia triste comunidad global lider internaciona...,[Acuerdo de París],"""Es un día triste para la comunidad global"": l...",Negative,0.982469
785556,De mal en peor | ONU advierte sobre crisis med...,mal peor | ONU advierte crisis medioambiental ...,mal peor onu advertir crisis medioambiental mo...,[ONU],De mal en peor | ONU advierte sobre crisis med...,Negative,0.982309
252013,¡Que horror! Esto es culpa de obrador (sarcasmo).,¡Que horror! culpa obrador (sarcasmo).,horror culpa obrador sarcasmo,[],¡Que horror! Esto es culpa de obrador (sarcasmo).,Negative,0.981910
267935,Es un triste espectáculo ver el desprendimient...,triste espectáculo desprendimiento hielos mile...,triste espectaculo desprendimiento hielo milen...,[],Es un triste espectáculo ver el desprendimient...,Negative,0.981205
...,...,...,...,...,...,...,...
873358,#EventosVirtuales No te pierdas esta gran char...,"#EventosVirtuales pierdas charla virtual ""Camb...",eventosvirtual pierdar charla virtual cambio c...,[https://bit.ly/38dTxu4:=:https://www.agrosavi...,#EventosVirtuales No te pierdas esta gran char...,Negative,0.344494
307875,"No esperes más, preséntanos tu proyecto #clima...","esperes más, preséntanos proyecto #climatekic_...",esperser mas presentanos proyecto climatekicac...,[],"No esperes más, preséntanos tu proyecto #clima...",Negative,0.344328
1221067,"""No vemos que en la Ciudad estén haciendo algo...","""No vemos Ciudad protegernos cambio climático ...",ciudad proteger cambio climatico avecinar,[],"""No vemos que en la Ciudad estén haciendo algo...",Negative,0.342644
250982,Los árboles por sí solos no pueden salvarnos d...,árboles salvarnos cambio climático reforestaci...,arbol salvar cambio climatico reforestacion ay...,[],Los árboles por sí solos no pueden salvarnos d...,Negative,0.338550


Now we need to extract each of the organizations alongside it's sentiment score. We will then loop through each, tallying up a total sentiment score and count.

Before we do that, we need to convert each value in the *organizations* column to a list.

In [41]:
# initialize sentiment dictionary
sentiment = {}

# loop through dataframe and extract org labels and sentiment scores into sentiment dictionary
for i, row in sub_sample.iterrows():
    # extract sentiment direction and score
    direction = row['sentiment']
    score = row['score']
    # loop through each label in organizations column
    for org in row['organizations']:
        # check if org label exists in sentiment dictionary already
        if org not in sentiment.keys():
            # if it doesn't, initialize new entry in dictionary
            sentiment[org] = {'Positive': [], 'Negative': [], 'Neutral': [],}
        # append positive/negative score to respective dictionary entry
        sentiment[org][direction].append(score)

In [42]:
sentiment['BOGOTÁ']

{'Positive': [0.6816926598548889,
  0.516435980796814,
  0.5245352387428284,
  0.5338535383343697,
  0.5480635017156601,
  0.5568593641122183,
  0.5665497779846191,
  0.5361912399530411,
  0.5423871815204621],
 'Negative': [],
 'Neutral': []}

Now we can loop through each organization entry in the sentiment dictionary and calculate an average positive, and average negative score:

**Question**: 
- *How can we include neutral sentiment?*

In [43]:
sentiment.keys()

dict_keys(['BOGOTÁ', 'Cambio Climático', 'RT', 'Parlamento británico', 'COP 25 sobre Cambio Climático', 'Comisión Ad Hoc', 'Comisiones Legislativas', 'ALIENTE', 'Alianza Energía y Territorio', 'Alianza Global Clima y Salud', 'Organización Mundial de la Salud', 'OMS', 'Razón', 'Unión de Pequeños Agricultores', 'Unión Europea', 'Radio Andalucía Información', 'CFE', 'PAN', 'MC', 'PRD', 'Senado de la República', 'PRI', 'Comisión Federal de Electricidad', 'International Surfing Association Surfrider Foundation Federación de Surf', 'Surfing Nation Magazine', 'AUSTRALIA', 'INCENDIOS', 'Sociedad Australiana de Animales', 'Universidad de Sydney', 'ACCIONA', 'Senior Policy Expert', 'European Desalination Society', 'Comité de Dirección de la', 'Fresh-Thoughts Consulting GmbH.', 'Estudios Internacional de Aqualia', 'La Federación Ambientalista Internacional', 'Comité de Dirección de Water Europe', 'Comisión Europea', 'Directiva', 'Wireless Innovative MMIC', 'OCDE', 'Comité Ejecutivo de Eureau', 'W

In [44]:
# initialize sentiment list
avg_sentiment = []

# loop through each organization
for org in sentiment.keys():
    # get number of positive and negative ratings
    freq = len(sentiment[org]['Positive']) + len(sentiment[org]['Negative'])
    # Ommit organizations without Positives and Negatives Scores
    if freq == 0:
        continue
    for direction in ['Positive', 'Negative']:
        # assign to variable for cleaner code
        score = sentiment[org][direction]
        # if there are no entries, set to 0
        if len(score) == 0:
            sentiment[org][direction] = 0.0
        else:
            # otherwise calculate total
            sentiment[org][direction] = sum(score)
    # now calculate total amount
    total = sentiment[org]['Positive'] - sentiment[org]['Negative']
    # and the average score
    avg = total/freq
    # add to sentiment list
    avg_sentiment.append({
        'entity': org,
        'positive': sentiment[org]['Positive'],
        'negative': sentiment[org]['Negative'],
        'frequency': freq,
        'score': avg
    })

In [45]:
sentiment_df = pd.DataFrame(avg_sentiment)
sentiment_df.head()

,entity,positive,negative,frequency,score
0,BOGOTÁ,5.006568,0.000000,9,0.556285
1,Cambio Climático,1935.531332,683.173584,4087,0.306425
2,RT,1.584514,4.136871,10,-0.255236
3,Parlamento británico,0.000000,3.646882,5,-0.729376
4,COP 25 sobre Cambio Climático,0.000000,0.627384,1,-0.627384


Immediately we can see we have a lot of entities which have appeared once in our dataset, and because of this their score will be pushed to one extreme or the other. We can filter out anything with less than or equal to a frequency of `100` to remove many of these instances:

In [46]:
sentiment_df = sentiment_df[sentiment_df['frequency'] > 100]
sentiment_df

,entity,positive,negative,frequency,score
1,Cambio Climático,1935.531332,683.173584,4087,0.306425
10,Organización Mundial de la Salud,46.357480,86.839093,211,-0.191856
11,OMS,74.693125,172.046119,381,-0.255520
14,Unión Europea,347.553320,158.944926,793,0.237842
16,CFE,29.022127,38.042800,109,-0.082758
...,...,...,...,...,...
2020,Ejecutivo,76.778078,45.911094,202,0.152807
2234,Medio Ambiente,60.525606,15.904163,121,0.368772
2242,Microsoft,37.436470,29.838657,107,0.071008
2306,CambioClimático,81.712578,54.719665,216,0.124967


In [47]:
sentiment_df.head(30)

,entity,positive,negative,frequency,score
1,Cambio Climático,1935.531332,683.173584,4087,0.306425
10,Organización Mundial de la Salud,46.357480,86.839093,211,-0.191856
11,OMS,74.693125,172.046119,381,-0.255520
14,Unión Europea,347.553320,158.944926,793,0.237842
16,CFE,29.022127,38.042800,109,-0.082758
17,PAN,31.271277,43.184041,120,-0.099273
21,PRI,44.029428,54.355552,161,-0.064137
37,Comisión Europea,74.010832,29.714744,167,0.265246
55,ONU,880.045140,1023.803766,3010,-0.047760
56,IPCC,98.135061,278.140639,612,-0.294127


# References

- [Hugging Face Bert](https://huggingface.co/docs/transformers/model_doc/bert)
- [Intro to Tokenizer for Bert](https://medium.com/@dhartidhami/understanding-bert-word-embeddings-7dc4d2ea54ca)
- [Hugging Face model DistilBert](https://huggingface.co/lxyuan/distilbert-base-multilingual-cased-sentiments-student?) 
- [Tokenizador de OpenAI](https://platform.openai.com/tokenizer)